In [2]:
# 1. INSTALAMOS LIBRERÍAS
!pip install -qU groq pypdf

import os
import re
from google.colab import userdata, drive
from groq import Groq
from pypdf import PdfReader

# 2. MONTAR GOOGLE DRIVE
drive.mount('/content/drive')

# 3. RUTA DEL PDF EN TU GOOGLE DRIVE
# (Asegúrate de cambiar esta ruta por la ubicación real de tu archivo en Drive)
RUTA_PDF = "/content/drive/MyDrive/BOE-A-1978-31229-consolidado.pdf"

# 4. CLIENTE DE GROQ (Lee la clave guardada en los Secretos de Colab)
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# 5. CARGAR EL PDF DIVIDIÉNDOLO POR ARTÍCULOS
def cargar_pdf_por_articulos(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo en la ruta: {path}. Revisa el nombre y la carpeta en Drive.")

    reader = PdfReader(path)
    texto_completo = ""

    # Extraemos todo el texto
    for page in reader.pages:
        t = page.extract_text()
        if t:
            texto_completo += t + "\n"

    # Dividimos el documento cada vez que encuentra la palabra "Artículo X."
    articulos = re.split(r'\n(?=Artículo\s+\d+\.)', texto_completo)

    # Limpiamos fragmentos vacíos o muy cortos
    chunks = [art.strip() for art in articulos if len(art.strip()) > 30]
    return chunks

try:
    base_de_conocimiento = cargar_pdf_por_articulos(RUTA_PDF)
    print(f"\n✅ PDF cargado con éxito. Se procesaron {len(base_de_conocimiento)} artículos/secciones.\n")
except Exception as e:
    print(f"\n❌ Error al cargar el PDF: {e}\n")

# 6. RECUPERADOR INTELIGENTE (RETRIEVAL)
def recuperar_contexto_relevante(pregunta, documentos):
    pregunta_lower = pregunta.lower()
    docs_relevantes = []

    # A) Buscar si hay un número de artículo en la pregunta (Ej: "artículo 14", "art 116")
    match_articulo = re.search(r'(?:artículo|articulo|art\.?)\s*(\d+)', pregunta_lower)

    if match_articulo:
        num_art = match_articulo.group(1)
        patron_art = f"artículo {num_art}."

        for doc in documentos:
            if patron_art in doc.lower():
                docs_relevantes.append((100, doc))  # Prioridad máxima

    # B) Búsqueda general por coincidencias de palabras clave
    palabras_pregunta = set(re.findall(r'\w+', pregunta_lower))
    palabras_ignorar = {"qué", "que", "cómo", "como", "sobre", "dice", "según", "segun", "cuál", "cual", "artículo", "articulo"}
    palabras_clave = {p for p in palabras_pregunta if len(p) > 3 and p not in palabras_ignorar}

    for doc in documentos:
        doc_lower = doc.lower()
        coincidencias = sum(1 for palabra in palabras_clave if palabra in doc_lower)
        if coincidencias > 0:
            docs_relevantes.append((coincidencias, doc))

    # Ordenamos de mayor a menor relevancia
    docs_relevantes.sort(key=lambda x: x[0], reverse=True)

    if not docs_relevantes:
        return "No se encontró información relevante en el documento para esta consulta."

    # Devolvemos los 3 fragmentos/artículos con mayor puntuación sin repetir
    mejores_docs = []
    vistos = set()
    for _, doc in docs_relevantes:
        if doc not in vistos:
            mejores_docs.append(doc)
            vistos.add(doc)
        if len(mejores_docs) >= 3:
            break

    return "\n\n--- ARTÍCULO RECUPERADO ---\n\n".join(mejores_docs)

# 7. BUCLE DEL CHATBOT RAG
historial = []

print("=== Chatbot RAG sobre la Constitución Listo (Escribe 'salir' para terminar) ===\n")

while True:
    pregunta = input("Tú: ")
    if pregunta.lower() in ["salir", "exit"]:
        break

    # R: Recuperación activa
    contexto_recuperado = recuperar_contexto_relevante(pregunta, base_de_conocimiento)

    # A: Prompt aumentado con el contexto legal
    prompt = f"""Eres un asistente jurídico experto. Responde a la pregunta utilizando ÚNICAMENTE la información extraída del documento oficial.
Si la información no aparece en el contexto, di exactamente: "No encuentro esa información en el documento proporcionado".

CONTEXTO EXTRAÍDO DEL DOCUMENTO:
{contexto_recuperado}

HISTORIAL PREVIO:
{historial}

PREGUNTA ACTUAL:
{pregunta}"""

    # G: Generación con Groq (Llama 3.3)
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
        )

        respuesta_texto = response.choices[0].message.content
        print(f"\nRespuesta:\n{respuesta_texto}\n")

        historial.append(f"Usuario: {pregunta}")
        historial.append(f"Asistente: {respuesta_texto}")

    except Exception as e:
        print(f"\nError al consultar el modelo: {e}\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ PDF cargado con éxito. Se procesaron 170 artículos/secciones.

=== Chatbot RAG sobre la Constitución Listo (Escribe 'salir' para terminar) ===

Tú: ¿Qué dice el artículo 14 sobre la igualdad?

Respuesta:
El artículo 14 establece que los españoles son iguales ante la ley, sin que pueda prevalecer discriminación alguna por razón de nacimiento, raza, sexo, religión, opinión o cualquier otra condición o circunstancia personal o social.

Tú: ¿Cómo se regula el estado de alarma, excepción y sitio según el artículo 116?

Respuesta:
Según el artículo 116, el estado de alarma, excepción y sitio se regula de la siguiente manera:

1. Una ley orgánica regulará estos estados y las competencias y limitaciones correspondientes.
2. El estado de alarma será declarado por el Gobierno mediante decreto acordado en Consejo de Ministros por un plazo máximo de 15 días, dando cue

KeyboardInterrupt: Interrupted by user